In [28]:
#imports
import os
import torch
import copy
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T

from gammanet.models import VGG16GammaNetV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [29]:
#Paths
input_dir = "/home/yentl/pytorch_gammanet/Images"
output_dir = "/home/yentl/pytorch_gammanet/Output_Images"
checkpoint_path = "/home/yentl/pytorch_gammanet/checkpoint_epoch_40.pt"

os.makedirs(output_dir, exist_ok=True)

In [30]:
#Inspect weight dimensions
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

config = checkpoint['config']['model']

model = VGG16GammaNetV2(config)
model.load_state_dict(checkpoint['model_state_dict'], strict=False)

model.to(device)
model.eval()

print(f"Loaded model from epoch {checkpoint['epoch']}")
print(f"Validation F1: {checkpoint.get('best_metric', 'N/A')}")

/home/yentl/anaconda3/envs/pygestalt_env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/yentl/anaconda3/envs/pygestalt_env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loaded model from epoch 40
Validation F1: 0.6260475346234085


In [31]:
#Overview
print(model)

VGG16GammaNetV2(
  (block1_conv): VGG16Block(
    (layers): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (block2_conv): VGG16Block(
    (layers): Sequential(
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
    )
  )
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (block3_conv): VGG16Block(
    (layers): Sequential(
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     

In [32]:
# INSPECT fGRU WEIGHTS
print("\n=== fGRU Weight Shapes ===")

for i in range(5):
    fgru = getattr(model, f"fgru_{i}")

    print(f"\nfGRU_{i}")
    print("W_exc:", list(fgru.W_exc.shape))
    print("W_inh:", list(fgru.W_inh.shape))


=== fGRU Weight Shapes ===

fGRU_0
W_exc: [64, 64, 3, 3]
W_inh: [64, 64, 3, 3]

fGRU_1
W_exc: [128, 128, 3, 3]
W_inh: [128, 128, 3, 3]

fGRU_2
W_exc: [256, 256, 3, 3]
W_inh: [256, 256, 3, 3]

fGRU_3
W_exc: [512, 512, 3, 3]
W_inh: [512, 512, 3, 3]

fGRU_4
W_exc: [512, 512, 3, 3]
W_inh: [512, 512, 3, 3]


## Meaning of dimensions

Example:

W_exc shape = [C, C, 3, 3]

This means:
- C = number of feature channels (e.g., 64, 128, 256)
- 3x3 = spatial neighborhood

Interpretation:

Each feature channel has:
→ connections to ALL other channels
→ AND spatial neighbors

So:

Each E neuron:
- is located at (x, y, channel)
- connects to:
    - nearby pixels (3x3)
    - all feature channels

This means E/I are NOT just per-pixel scalars.
They are spatial + feature-aware units.

In [33]:
# INSPECT RECURRENT TIMESTEPS
print("\n=== RECURRENT TIMESTEPS ===")

for i in range(5):

    fgru = getattr(model, f"fgru_{i}")

    if hasattr(fgru, "timesteps"):

        print(f"fgru_{i} timesteps:", fgru.timesteps)

    elif hasattr(fgru, "time_steps"):

        print(f"fgru_{i} timesteps:", fgru.time_steps)

    else:

        print(f"fgru_{i}: timestep attribute not found")


=== RECURRENT TIMESTEPS ===
fgru_0: timestep attribute not found
fgru_1: timestep attribute not found
fgru_2: timestep attribute not found
fgru_3: timestep attribute not found
fgru_4: timestep attribute not found


In [34]:
# IMAGE TRANSFORM
transform = T.Compose([

    T.Resize((256,256)),

    T.ToTensor(),

    T.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [35]:
# GLOBAL STORAGE
activations = {}

ei_states = {}

temporal_states = {}

layer_summary = {}

hook_handles = []

In [36]:
def reset_hidden_states(self):

    self.h0_exc = None
    self.h1_exc = None
    self.h2_exc = None
    self.h3_exc = None
    self.h4_exc = None

    if hasattr(self, "h0_inh"):

        self.h0_inh = None
        self.h1_inh = None
        self.h2_inh = None
        self.h3_inh = None
        self.h4_inh = None

    # top-down states
    if hasattr(self, "td_h0_exc"):

        self.td_h0_exc = None
        self.td_h1_exc = None
        self.td_h2_exc = None
        self.td_h3_exc = None

    if hasattr(self, "td_h0_inh"):

        self.td_h0_inh = None
        self.td_h1_inh = None
        self.td_h2_inh = None
        self.td_h3_inh = None

    # temporal activity logging
    self.temporal_activity = {}

In [37]:
#Hooks (Conv + fGRU)
def get_activation(name):
    def hook(module, input, output):
        if isinstance(output, tuple):
            output = output[0]
        activations[name] = output.detach()
    return hook

def get_fgru_state(name):
    def hook(module, input, output):
        if hasattr(module, "excitation"):
            ei_states[name+"_E"] = module.excitation.detach()
        if hasattr(module, "inhibition"):
            ei_states[name+"_I"] = module.inhibition.detach()
    return hook

def get_temporal_hook(name):

    def hook(module, input, output):

        if name not in temporal_states:
            temporal_states[name] = []

        if isinstance(output, tuple):
            out = output[0]
        else:
            out = output

        temporal_states[name].append(
            out.detach().cpu()
        )

    return hook

# REGISTER HOOKS
# VGG layers
vgg_blocks = {
    "block1": model.block1_conv,
    "block2": model.block2_conv,
    "block3": model.block3_conv,
    "block4": model.block4_conv,
    "block5": model.block5_conv,
}

#Conv layers
for block_name, block in vgg_blocks.items():
    for i, layer in enumerate(block.layers):
        h = layer.register_forward_hook(get_activation(f"{block_name}_layer{i}"))

        hook_handles.append(h)

# fGRU layers
for i in range(5):

    fgru = getattr(model, f"fgru_{i}")

    h1 = fgru.register_forward_hook(
        get_activation(f"fgru_{i}")
    )

    h2 = fgru.register_forward_hook(
        get_fgru_state(f"fgru_{i}")
    )

    h3 = fgru.register_forward_hook(
        get_temporal_hook(f"fgru_{i}")
    )

    hook_handles.extend([h1, h2, h3])

# E/I hooks
# for i in range(5):
#     getattr(model, f"fgru_{i}").register_forward_hook(
#         get_fgru_state(f"fgru_{i}")
#     )

In [38]:
# STIMULUS GENERATORS

def create_line_image(size=256):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    cv2.line(img, (20, size//2), (size-20, size//2), (255,255,255), 2)
    return img

def create_curve_image(size=256):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    center = (size//2, size//2)
    cv2.ellipse(img, center, (80,80), 0, 0, 180, (255,255,255), 2)
    return img

def create_jittered_line(size=256, jitter=0):

    img = np.zeros((size, size, 3), dtype=np.uint8)

    x_positions = np.linspace(20, size-20, 30).astype(int)

    y_base = size // 2

    points = []

    for x in x_positions:

        y = y_base + np.random.randint(-jitter, jitter+1)

        points.append((x,y))

    for i in range(len(points)-1):

        cv2.line(img, points[i], points[i+1], (255,255,255), 2)

    return img

In [39]:
# BASIC ANALYSIS FUNCTIONS
def compute_stats(tensor):

    return {

        "mean": tensor.mean().item(),
        "std": tensor.std().item(),
        "min": tensor.min().item(),
        "max": tensor.max().item()
    }


def compute_activation_energy(fmap):

    return torch.mean(
        torch.abs(fmap)
    ).item()

def compute_sparsity(fmap, threshold=0.01):

    sparsity = (fmap.abs() < threshold).float().mean().item()

    return sparsity


def compute_variance(fmap):

    return fmap.var().item()


def compute_peak_activation(fmap):

    return fmap.max().item()


def compute_channel_selectivity(fmap):

    fmap = fmap.squeeze(0)

    mean_response = fmap.view(fmap.shape[0], -1).mean(dim=1)

    peak_response = fmap.view(fmap.shape[0], -1).max(dim=1)[0]

    variance_response = fmap.view(fmap.shape[0], -1).var(dim=1)

    return (mean_response, peak_response, variance_response)

In [40]:
# HEATMAP FUNCTIONS
# def create_activation_heatmap(fmap):

#     fmap = fmap.squeeze(0)

#     heatmap = torch.mean(
#         torch.abs(fmap),
#         dim=0
#     )

#     heatmap = heatmap.cpu().numpy()

#     heatmap -= heatmap.min()

#     heatmap /= (heatmap.max() + 1e-8)

#     return heatmap

def create_top_channel_heatmap(fmap):

    fmap = fmap.squeeze(0)

    peak_response = fmap.view(
        fmap.shape[0],
        -1
    ).max(dim=1)[0]

    best_channel = torch.argmax(
        peak_response
    ).item()

    heatmap = fmap[best_channel].cpu().numpy()

    heatmap -= heatmap.min()

    heatmap /= (heatmap.max() + 1e-8)

    return heatmap, best_channel


def save_heatmap_overlay(
    original_image,
    heatmap,
    save_path,
    title=None
):

    plt.figure(figsize=(6,6))

    plt.imshow(original_image)

    plt.imshow(
        heatmap,
        cmap='jet',
        alpha=0.5
    )

    if title is not None:
        plt.title(title)

    plt.axis("off")

    plt.tight_layout()

    plt.savefig(save_path)

    plt.close()

In [41]:
# FEATURE MAP VISUALIZATION
def save_feature_maps(tensor, layer_name, output_folder, num_channels=6):

    fmap = tensor.squeeze(0)
    n = min(num_channels, fmap.shape[0])

    plt.figure(figsize=(12,4))

    for i in range(n):
        plt.subplot(1,n,i+1)
        plt.imshow(fmap[i].cpu(), cmap="viridis")
        plt.axis("off")

    plt.suptitle(f"{layer_name} (first {n} channels)")
    plt.tight_layout()

    plt.savefig(os.path.join(output_folder, f"{layer_name}.png"))
    plt.close()

In [42]:
# ============================================================
# TEMPORAL ANALYSIS
# ============================================================

def analyze_temporal_dynamics(
    model,
    image,
    output_folder
):

    temporal_states = model.temporal_activity

    temporal_dir = os.path.join(
        output_folder,
        "temporal"
    )

    os.makedirs(temporal_dir, exist_ok=True)

    # --------------------------------------------------------
    # TEMPORAL ENERGY CURVES
    # --------------------------------------------------------

    plt.figure(figsize=(8,5))

    for layer_name, states in temporal_states.items():

        energies = []

        for t, tensor in enumerate(states):

            energy = torch.mean(
                torch.abs(tensor)
            ).item()

            energies.append(energy)

            print(
                f"{layer_name} "
                f"t={t} "
                f"energy={energy:.4f}"
            )

        plt.plot(
            range(len(energies)),
            energies,
            marker='o',
            label=layer_name
        )

    plt.xlabel("Timestep")

    plt.ylabel("Activation energy")

    plt.title("Temporal Recurrent Dynamics")

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            temporal_dir,
            "temporal_dynamics.png"
        )
    )

    plt.close()

    # --------------------------------------------------------
    # TEMPORAL HEATMAPS
    # --------------------------------------------------------

    heatmap_dir = os.path.join(
        temporal_dir,
        "heatmaps"
    )

    os.makedirs(heatmap_dir, exist_ok=True)

    for layer_name, states in temporal_states.items():

        for t, tensor in enumerate(states):

            fmap = tensor.squeeze(0)

            peak_response = fmap.view(
                fmap.shape[0],
                -1
            ).max(dim=1)[0]

            best_channel = torch.argmax(
                peak_response
            ).item()

            heatmap = fmap[best_channel].numpy()

            heatmap -= heatmap.min()

            heatmap /= (
                heatmap.max() + 1e-8
            )

            save_heatmap_overlay(
                image,
                heatmap,
                os.path.join(
                    heatmap_dir,
                    f"{layer_name}_t{t}.png"
                ),
                title=f"{layer_name} t={t}"
            )

In [43]:
# E/I BALANCE
def analyze_ei_balance():

    print("\n=== E/I BALANCE ===")

    for key in ei_states:

        tensor = ei_states[key]

        mean = tensor.mean().item()

        std = tensor.std().item()

        peak = tensor.max().item()

        sparsity = compute_sparsity(tensor)

        print(
            f"{key}: "
            f"mean={mean:.4f}, "
            f"std={std:.4f}"
            f"peak={peak:.4f}"
            f"sparsity{sparsity:.4f}"
        )

In [44]:
# TEMPORAL ANALYSIS
def compute_temporal_energy():

    print("\n=== TEMPORAL DYNAMICS ===")

    temporal_energy = {}

    for layer_name, states in temporal_states.items():

        energies = []

        for t, tensor in enumerate(states):

            energy = torch.mean(torch.abs(tensor)).item()

            energies.append(energy)

            print(f"{layer_name} "
                f"t={t} "
                f"energy={energy:.4f}"
            )

        temporal_energy[layer_name] = energies

    return temporal_energy


def plot_temporal_dynamics(

    temporal_energy,
    save_path
):

    plt.figure(figsize=(8,5))

    for layer_name, energies in temporal_energy.items():

        plt.plot(range(len(energies)), energies, marker='o', label=layer_name)

    plt.xlabel("Recurrent timestep")

    plt.ylabel("Activation energy")

    plt.title("Temporal Recurrent Dynamics")

    plt.legend()

    plt.tight_layout()

    plt.savefig(save_path)

    plt.close()


def save_temporal_heatmaps( temporal_states, original_image, output_folder):

    os.makedirs(output_folder, exist_ok=True)

    for layer_name, states in temporal_states.items():

        for t, tensor in enumerate(states):

            fmap = tensor.squeeze(0)

            heatmap = torch.mean(torch.abs(fmap), dim=0).numpy()

            heatmap -= heatmap.min()

            heatmap /= (heatmap.max() + 1e-8)

            plt.figure(figsize=(6,6))

            plt.imshow(original_image)

            plt.imshow(heatmap, cmap='jet', alpha=0.5)

            plt.title(f"{layer_name} timestep={t}")

            plt.axis("off")

            plt.tight_layout()

            plt.savefig(

                os.path.join(output_folder, f"{layer_name}_t{t}.png"))

            plt.close()


def save_temporal_difference_maps(temporal_states, output_folder):

    os.makedirs(output_folder, exist_ok=True)

    for layer_name, states in temporal_states.items():

        for t in range(1, len(states)):

            prev = states[t-1]

            curr = states[t]

            diff = torch.mean(torch.abs(curr - prev), dim=1).squeeze(0)

            diff = diff.numpy()

            plt.figure(figsize=(5,5))

            plt.imshow(diff, cmap='inferno')

            plt.title(f"{layer_name} Δ t{t-1}->{t}")

            plt.colorbar()

            plt.tight_layout()

            plt.savefig(

                os.path.join(output_folder, f"{layer_name}_diff_{t}.png"))

            plt.close()

In [45]:
# DATASET SUMMARY
def dataset_summary(input_dir):

    image_files = [

        f for f in os.listdir(input_dir)

        if f.lower().endswith(
            (".png",".jpg",".jpeg",".bmp",".tiff")
        )
    ]

    print("\n=== DATASET SUMMARY ===")

    print("Total images:", len(image_files))


dataset_summary(input_dir)


=== DATASET SUMMARY ===
Total images: 2


In [46]:
# SHAPE SELECTIVITY TEST
def analyze_shape_response(img, label):

    image = Image.fromarray(img)
    input_tensor = transform(image).unsqueeze(0).to(device)

    activations.clear()

    # RESET RECURRENT STATES
    if hasattr(model, "reset_hidden_states"):
        model.reset_hidden_states()

    with torch.no_grad():
        _ = model(input_tensor)

    print(f"\n=== {label} ===")

    for name, fmap in activations.items():
        fmap = fmap.squeeze(0)
        channel_means = fmap.view(fmap.shape[0], -1).mean(dim=1)
        topk = torch.topk(channel_means, k=5)

        print(f"{name} → top channels: {topk.indices.tolist()}")

line_img = create_line_image()

curve_img = create_curve_image()

analyze_shape_response(line_img, "STRAIGHT LINE")

analyze_shape_response(curve_img, "CURVE")


=== STRAIGHT LINE ===
block1_layer0 → top channels: [44, 12, 10, 37, 60]
block1_layer1 → top channels: [44, 12, 10, 37, 60]
block1_layer2 → top channels: [29, 26, 38, 43, 60]
block1_layer3 → top channels: [29, 26, 38, 43, 60]
fgru_0 → top channels: [31, 50, 39, 2, 4]
block2_layer0 → top channels: [34, 116, 117, 92, 101]
block2_layer1 → top channels: [34, 116, 117, 92, 101]
block2_layer2 → top channels: [26, 44, 22, 28, 49]
block2_layer3 → top channels: [26, 44, 22, 28, 49]
fgru_1 → top channels: [11, 99, 40, 28, 65]
block3_layer0 → top channels: [99, 9, 219, 62, 197]
block3_layer1 → top channels: [99, 9, 219, 62, 197]
block3_layer2 → top channels: [71, 234, 169, 164, 49]
block3_layer3 → top channels: [71, 234, 169, 164, 49]
block3_layer4 → top channels: [96, 206, 136, 53, 119]
block3_layer5 → top channels: [96, 206, 136, 53, 119]
fgru_2 → top channels: [42, 238, 95, 101, 156]
block4_layer0 → top channels: [419, 13, 234, 308, 490]
block4_layer1 → top channels: [419, 13, 234, 308, 490]


In [47]:
# TOP-DOWN MANIPULATION
original_model = copy.deepcopy(model)

test_image = Image.open(

    os.path.join(input_dir, os.listdir(input_dir)[0])

).convert("RGB")

input_tensor = transform(test_image).unsqueeze(0).to(device)

# RESET RECURRENT STATES
if hasattr(model, "reset_hidden_states"):
    model.reset_hidden_states()

with torch.no_grad():

    out_full = model(input_tensor)


# Example manipulation:
# Replace higher fGRU outputs with zeros

for i in range(3,5):

    fgru = getattr(model, f"fgru_{i}")

    def dummy_forward(*args, **kwargs):

        x = args[0]

        return (
            torch.zeros_like(x),
            torch.zeros_like(x),
            None
        )

    fgru.forward = dummy_forward

# RESET RECURRENT STATES
if hasattr(model, "reset_hidden_states"):
    model.reset_hidden_states()

with torch.no_grad():

    out_no_td = model(input_tensor)


print(

    "\nTop-down difference:",

    torch.abs(out_full - out_no_td).mean().item()
)

# Restore model
model = original_model

model.eval()

model.to(device)


Top-down difference: 6.4559221267700195


VGG16GammaNetV2(
  (block1_conv): VGG16Block(
    (layers): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (block2_conv): VGG16Block(
    (layers): Sequential(
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
    )
  )
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (block3_conv): VGG16Block(
    (layers): Sequential(
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     

In [48]:
# MAIN DATASET LOOP
layer_summary = {}

for image_name in os.listdir(input_dir):

    if not image_name.lower().endswith((".png",".jpg",".jpeg",".bmp",".tiff")):
        continue

    print(f"\nProcessing {image_name}")

    base_name = os.path.splitext(image_name)[0]

    image_output_dir = os.path.join(output_dir, base_name)
    featuremap_dir = os.path.join(image_output_dir, "featuremaps")
    temporal_dir = os.path.join(image_output_dir, "temporal")

    os.makedirs(featuremap_dir, exist_ok=True)
    os.makedirs(temporal_dir, exist_ok=True)

    # Load image
    image = Image.open(os.path.join(input_dir, image_name)).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    # Clear states
    activations.clear()
    ei_states.clear()
    temporal_states.clear()

    # ============================================================
    # RESET RECURRENT STATES
    # ============================================================

    if hasattr(model, "reset_hidden_states"):
        model.reset_hidden_states()

    # Forward pass
    with torch.no_grad():
        output = model(input_tensor)
        edge_prob = torch.sigmoid(output)

    edge_map = edge_prob[0,0].cpu().numpy()

    # DEBUG NUMERICAL STABILITY

    for key, states in model.temporal_activity.items():

        for t, tensor in enumerate(states):

            has_nan = torch.isnan(tensor).any().item()
            has_inf = torch.isinf(tensor).any().item()

            print(
                f"{key} t={t} "
                f"nan={has_nan} "
                f"inf={has_inf}"
            )

            if has_nan or has_inf:

                print(
                    "MIN:",
                    tensor.min().item()
                )

                print(
                    "MAX:",
                    tensor.max().item()
                )

    # Temporal analysis
    # Get recurrent temporal states
    temporal_states = model.temporal_activity

    plt.figure(figsize=(8,5))

    for layer_name, states in temporal_states.items():

        energies = []

        for t, tensor in enumerate(states):

            energy = torch.mean(
                torch.abs(tensor)
            ).item()

            energies.append(energy)

            print(
                f"{layer_name} "
                f"t={t} "
                f"energy={energy:.4f}"
            )

        plt.plot(
            range(len(energies)),
            energies,
            marker='o',
            label=layer_name
        )

    plt.xlabel("Timestep")

    plt.ylabel("Activation energy")

    plt.title("Temporal Recurrent Dynamics")

    plt.legend()

    plt.savefig(
        os.path.join(
            temporal_dir,
            "temporal_dynamics.png"
        )
    )

    plt.close()

    # Temporal analysis
    analyze_temporal_dynamics(model,image,image_output_dir)

    # E/I analysis
    analyze_ei_balance()    

    # -----------------------------
    # EDGE STATISTICS
    # -----------------------------
    print("\nEdge statistics:")
    print("Mean:", edge_map.mean())
    print("Std:", edge_map.std())
    print("Max:", edge_map.max())

    # -----------------------------
    # Edge map
    # -----------------------------
    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)

    plt.imshow(image)

    plt.title("Input")

    plt.axis("off")

    plt.subplot(1,2,2)

    plt.imshow(edge_map, cmap="gray")

    plt.title("Edge map")

    plt.axis("off")

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            image_output_dir,
            "edge_map.png"
        )
    )

    plt.close()

    # -----------------------------
    # THRESHOLD ANALYSIS
    # -----------------------------
    thresholds = np.arange(0.1, 1.1, 0.1)
    edge_ratios = []

    fig, axes = plt.subplots(2,5, figsize=(15,6))

    for idx, t in enumerate(thresholds):
        binary = edge_map > t
        ratio = binary.sum() / binary.size
        edge_ratios.append(ratio)

        axes[idx//5, idx%5].imshow(binary, cmap="gray")
        axes[idx//5, idx%5].set_title(f"t={t:.1f}\nr={ratio:.2f}")
        axes[idx//5, idx%5].axis("off")

    plt.savefig(os.path.join(image_output_dir, "threshold_overview.png"))
    plt.close()

    # Threshold Curve
    plt.figure()
    plt.plot(thresholds, edge_ratios, marker="o")
    plt.title("Edge Ratio Curve")
    plt.xlabel("Threshold")
    plt.ylabel("Edge Ratio")
    plt.savefig(os.path.join(image_output_dir, "threshold_curve.png"))
    plt.close()

    # INPUT / OUTPUT OVERVIEW
    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1)
    plt.imshow(image)
    plt.title("Input")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(edge_map, cmap="gray")
    plt.title("Edge map")
    plt.axis("off")

    plt.savefig(os.path.join(image_output_dir, f"edges_overview_{base_name}.png"))
    plt.close()

    # -----------------------------
    # FEATURE MAPS + STATS
    # -----------------------------
    for name, fmap in activations.items():

        print(f"\n{name}: "
            f"{list(fmap.shape)}"
        )

        stats = compute_stats(fmap)

        print("Stats:", stats)


        # ----------------------------------------------------
        # Activation energy
        # ----------------------------------------------------

        energy = compute_activation_energy(fmap)

        print(f"Energy: {energy:.4f}")

        sparsity = compute_sparsity(fmap)

        print(f"Sparsity: {sparsity:.4f}")

        variance = compute_variance(fmap)

        print(f"Variance: {variance:.4f}")

        peak = compute_peak_activation(fmap)

        print(f"Peak: {peak:.4f}")

        # Store population stats
        if name not in layer_summary:

            layer_summary[name] = []

        layer_summary[name].append(energy)


        # ----------------------------------------------------
        # Channel selectivity
        # ----------------------------------------------------

        (mean_response, peak_response, variance_response) = compute_channel_selectivity(fmap)

        top_channels = torch.topk(peak_response, k=5)

        print("Top selective channels:")

        for idx in top_channels.indices:

            idx = idx.item()

            print(
                f"Channel {idx} "
                f"mean={mean_response[idx]:.4f} "
                f"peak={peak_response[idx]:.4f} "
                f"variance={variance_response[idx]:.4f}"
            )

        # ----------------------------------------------------
        # Feature maps
        # ----------------------------------------------------

        save_feature_maps(fmap, name, featuremap_dir)


        # ----------------------------------------------------
        # Heatmap overlay
        # ----------------------------------------------------

        heatmap, best_channel = create_top_channel_heatmap(fmap)

        save_heatmap_overlay(
            image,
            heatmap,
            os.path.join(
                featuremap_dir,
                f"{name}_top_channel.png"
            ),
            title=f"{name} channel={best_channel}"
        )
    
    # ========================================================
    # TEMPORAL ANALYSIS
    # ========================================================

    temporal_energy = compute_temporal_energy()


    plot_temporal_dynamics(

        temporal_energy,

        os.path.join(temporal_dir, "temporal_dynamics.png"        )
    )


    save_temporal_heatmaps(

        temporal_states,

        image,

        os.path.join( temporal_dir, "heatmaps")
    )


    save_temporal_difference_maps(

        temporal_states,

        os.path.join(temporal_dir, "difference_maps")
    )


Processing straight_low_BL_0_J000_013.png
h0_exc t=0 nan=False inf=False
h0_exc t=1 nan=False inf=False
h0_exc t=2 nan=False inf=False
h0_exc t=3 nan=False inf=False
h1_exc t=0 nan=False inf=False
h1_exc t=1 nan=False inf=False
h1_exc t=2 nan=False inf=False
h1_exc t=3 nan=False inf=False
h2_exc t=0 nan=False inf=False
h2_exc t=1 nan=False inf=False
h2_exc t=2 nan=False inf=False
h2_exc t=3 nan=False inf=False
h3_exc t=0 nan=False inf=False
h3_exc t=1 nan=False inf=False
h3_exc t=2 nan=False inf=False
h3_exc t=3 nan=False inf=False
h4_exc t=0 nan=False inf=False
h4_exc t=1 nan=False inf=False
h4_exc t=2 nan=False inf=False
h4_exc t=3 nan=False inf=False
h0_exc t=0 energy=0.5426
h0_exc t=1 energy=0.8581
h0_exc t=2 energy=1.2254
h0_exc t=3 energy=1.6965
h1_exc t=0 energy=0.4078
h1_exc t=1 energy=0.7127
h1_exc t=2 energy=0.9591
h1_exc t=3 energy=1.2521
h2_exc t=0 energy=0.5360
h2_exc t=1 energy=0.7869
h2_exc t=2 energy=1.1984
h2_exc t=3 energy=1.8081
h3_exc t=0 energy=0.6143
h3_exc t=1 e

In [49]:
# ============================================================
# POPULATION SUMMARY
# ============================================================

print("\n=== POPULATION SUMMARY ===")

for layer in layer_summary:

    values = np.array(layer_summary[layer])

    print(

        f"{layer}: "

        f"mean={values.mean():.4f} "

        f"std={values.std():.4f}"
    )


=== POPULATION SUMMARY ===
block1_layer0: mean=0.4042 std=0.0724
block1_layer1: mean=0.3909 std=0.0678
block1_layer2: mean=0.5509 std=0.1843
block1_layer3: mean=0.5416 std=0.1796
fgru_0: mean=1.7720 std=0.1191
block2_layer0: mean=0.9226 std=0.0068
block2_layer1: mean=0.9056 std=0.0191
block2_layer2: mean=1.1208 std=0.0501
block2_layer3: mean=1.0910 std=0.0735
fgru_1: mean=1.2700 std=0.0239
block3_layer0: mean=0.4455 std=0.0084
block3_layer1: mean=0.4406 std=0.0092
block3_layer2: mean=1.0216 std=0.0395
block3_layer3: mean=1.0097 std=0.0443
block3_layer4: mean=0.5989 std=0.0011
block3_layer5: mean=0.5964 std=0.0014
fgru_2: mean=1.8103 std=0.0423
block4_layer0: mean=0.6524 std=0.0011
block4_layer1: mean=0.6498 std=0.0013
block4_layer2: mean=1.2338 std=0.0108
block4_layer3: mean=1.2003 std=0.0106
block4_layer4: mean=0.5934 std=0.0158
block4_layer5: mean=0.5707 std=0.0143
fgru_3: mean=25.3276 std=8.4084
block5_layer0: mean=44.7874 std=16.3511
block5_layer1: mean=42.2688 std=15.6309
block5_

In [50]:
# ============================================================
# JITTER EXPERIMENT
# ============================================================

print("\n=== JITTER EXPERIMENT ===")

jitter_levels = [0,2,4,6,8,10,12,16]

scores = []

for jitter in jitter_levels:

    img = create_jittered_line(jitter=jitter)

    image = Image.fromarray(img)

    input_tensor = transform(image).unsqueeze(0).to(device)

    # RESET RECURRENT STATES
    if hasattr(model, "reset_hidden_states"):
        model.reset_hidden_states()

    with torch.no_grad():

        output = model(input_tensor)

        prob = torch.sigmoid(output)

    detection_score = prob.mean().item()

    scores.append(detection_score)

    print(f"Jitter={jitter} "
        f"score={detection_score:.4f}")


plt.figure()

plt.plot(
    jitter_levels,
    scores,
    marker='o'
)

plt.xlabel("Jitter")

plt.ylabel("Detection score")

plt.title("Contour Sensitivity Curve")

plt.savefig(

    os.path.join(
        output_dir,
        "jitter_sensitivity.png"
    )
)

plt.close()



=== JITTER EXPERIMENT ===
Jitter=0 score=0.0087
Jitter=2 score=0.0203
Jitter=4 score=0.0261
Jitter=6 score=0.0319
Jitter=8 score=0.0299
Jitter=10 score=0.0341
Jitter=12 score=0.0335
Jitter=16 score=0.0419


In [51]:
# ============================================================
# REMOVE HOOKS
# ============================================================

for h in hook_handles:

    h.remove()


print("\n=== ANALYSIS COMPLETE ===")


=== ANALYSIS COMPLETE ===


## Meaning of shapes?

Example:

Exc shape: [1, 64, 256, 256]

This means:

- 1 = batch
- 64 = number of E neurons per pixel
- 256x256 = spatial map

So:

At each pixel:
→ you have 64 excitatory neurons

Same for inhibitory.

So total E neurons:
64 × 256 × 256 ≈ 4 million neurons

These neurons are NOT independent:
They are connected via convolution:

→ spatial neighbors interact (3x3)
→ feature channels interact

This is how contours can "link" across space.